# Corrigé : Chapitre 2

*MLSMM2151 Data Science · Séances 1 et 2 · Corentin Vande Kerckhove*

Corrigé des hack-times des deux notebooks du chapitre 2 (notebook 1 en
séance 1, notebook 2 en séance 2, suivi du tutoriel du devoir 1). Ces exercices sont
manipulatoires : le corrigé donne le code, mais surtout ce qu'il fallait
observer.

## Notebook 1, hack-time 1 : ce qui manque dans `requirements.txt`

In [ ]:
# On teste la présence d'une bibliothèque sans faire planter le notebook si elle manque.
for nom in ["xgboost", "seaborn"]:
    try:
        module = __import__(nom)
        print(f"{nom:10} présent, version {module.__version__}")
    except ImportError:
        print(f"{nom:10} absent")

Sur Colab, les deux sont présents : Google les préinstalle, avec des centaines
d'autres paquets. C'est pour cela que « ça tourne » est vrai sur Colab, et faux
sur la machine du correcteur, qui installe `requirements.txt` et rien d'autre.

**Ce qu'il faut ajouter** : `xgboost>=2.0` et `seaborn>=0.13`. La borne
inférieure dit « au moins cette version », celle avec laquelle le code a été
écrit et testé.

**Pourquoi « ça tourne chez moi » n'est pas un argument.** Parce que le but
d'un `requirements.txt` est précisément de rendre « chez moi » inutile : la
liste doit suffire à n'importe qui, sur n'importe quelle machine. Un notebook
qui ne tourne que dans l'environnement de son auteur n'est pas reproductible,
et la grille des devoirs évalue la reproductibilité.

### Le défi optionnel : tout mettre dans `requirements.txt` ?

`!pip list | wc -l` renvoie plusieurs centaines de lignes. Non, on ne les met
pas toutes : on liste **ce que le code importe**, et seulement cela. Les
dépendances de ces paquets (numpy pour pandas, par exemple) sont installées
automatiquement. Une liste exhaustive figerait des centaines de versions sans
rapport avec votre projet et casserait à la première mise à jour de Colab.

## Notebook 1, hack-time 2 : notebook ou `.py` ?

| Bout de code | Dans un `.py` ? | Pourquoi |
|---|---|---|
| `nettoyer_prix`, copiée dans trois notebooks | **Oui** | Réutilisée, donc à écrire une fois ; testable ; un diff sur un `.py` montre exactement ce qui a changé, un diff sur trois notebooks non |
| `annonces.head()` | **Non** | Exploration pure : on regarde, on ne garde pas |
| Les trente lignes de la figure 2 | **Non en l'état, oui si elle revient** | Si la figure est produite une fois pour le rapport, le notebook suffit. Si elle est régénérée à chaque mise à jour des données, c'est une fonction `figure_2(donnees)` dans un `.py` |

**Pourquoi `sujet.py` est un fichier `.py`.** Trois raisons, dans l'ordre
d'importance. Il est identique pour tous : une seule copie, et non une cellule
que chacun pourrait modifier sans s'en rendre compte. Le correcteur le rejoue
tel quel pour vérifier votre variante. Et toute modification, même d'un
caractère, apparaît dans un diff lisible, ce qui n'est pas le cas d'une cellule
de notebook.

### Le défi optionnel : importer `src/nettoyage.py` depuis Colab

Dans un projet cloné, `from src.nettoyage import nettoyer_prix` suffit. Mais un
notebook ouvert depuis GitHub dans Colab arrive **seul** : rien d'autre n'est
cloné, et l'import échoue avec `ModuleNotFoundError`. Il faut d'abord récupérer
le fichier, exactement comme le fait la cellule `!curl ... donnees.py` du
cas 2, ou cloner le dépôt entier avec `!git clone`. C'est la raison d'être de
cette cellule, que vous retrouverez dans chaque devoir.

## Notebook 1, hack-time 3 : faire circuler un fichier par OneDrive

### Étape 1 : exporter un extrait

In [ ]:
import pandas as pd

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"
annonces = pd.read_csv(f"{BASE}/airbnb_bruxelles_raw.csv", low_memory=False)

extrait = annonces.head(50)
extrait.to_csv("extrait.csv", index=False)

print(f"Écrit : extrait.csv, {len(extrait)} lignes")

`index=False` n'est pas décoratif. Sans lui, pandas écrit l'index du DataFrame
comme première colonne, sans nom. Au rechargement, vous récupérez une colonne
parasite `Unnamed: 0`, et si vous faites l'aller-retour plusieurs fois, vous en
accumulez plusieurs.

### Étapes 2 et 3 : télécharger, déposer, partager

Dans Colab, panneau `Fichiers` à gauche, clic droit sur `extrait.csv`,
`Télécharger`. Puis dépôt sur OneDrive, `Partager`, `Copier le lien`.

Le lien copié ressemble à ceci :

```
https://uclouvain-my.sharepoint.com/:x:/g/personal/xxx/EaBcDeF...?e=aBcDeF
```

Ce lien pointe vers une **page web** de prévisualisation, pas vers le fichier.
D'où la conversion.

In [ ]:
def lien_direct(lien_partage):
    """Convertit un lien de partage OneDrive en lien de téléchargement direct."""
    return lien_partage.split("?")[0] + "?download=1"


exemple = "https://uclouvain-my.sharepoint.com/:x:/g/personal/xxx/EaBcDeF?e=aBcDeF"
print(lien_direct(exemple))

### Étape 4 : recharger

```python
donnees = pd.read_csv(lien_direct(mon_lien))
print(len(donnees))       # doit afficher 50
```

**Si cela échoue**, deux causes possibles, et il faut savoir les distinguer.

**Le partage est restreint.** L'UCLouvain peut limiter les liens aux membres de
l'institution. Colab n'est pas authentifié à l'UCLouvain : le serveur lui renvoie
une page de connexion. C'est un réglage administratif, pas une erreur de votre
part. La solution de repli est le Google Drive, montré dans le même notebook.

**Le `?download=1` a été oublié.** Voir ci-dessous, c'est le défi optionnel.

### Le défi optionnel : que se passe-t-il sans `?download=1` ?

Le serveur renvoie une **page HTML** de prévisualisation. `pd.read_csv` reçoit
donc du HTML et essaie de le lire comme du CSV. Selon la page, vous obtenez
soit une `ParserError`, soit, plus vicieux, un DataFrame d'une seule colonne
contenant des fragments de balises.

Le second cas est le plus dangereux : **il ne lève aucune erreur**. Un code qui
« marche » et produit des données absurdes est bien pire qu'un code qui plante.

Le réflexe qui coûte deux secondes et qui vous évitera des heures :

```python
donnees = pd.read_csv(lien)
print(donnees.shape)      # est-ce bien 50 lignes et 38 colonnes ?
donnees.head(2)           # est-ce que cela ressemble à mes données ?
```

**Vérifiez toujours la forme de ce que vous venez de charger.** C'est valable
pour ce hack-time, et pour tout le reste du cours.

## Notebook 2, hack-time 1 : réécrire trois messages de commit

| Avant | Après |
|---|---|
| `modif` | `Convertit price de texte en nombre` |
| `devoir fini` | `Ajoute le traitement du temps (partie 3)` **puis** `Rédige REFLEXION.md` |
| `oups` | `Retire le CSV commité par erreur et l'ajoute au .gitignore` |

Le deuxième aurait dû être deux commits : deux intentions différentes, et
l'historique doit montrer que la réflexion a été écrite séparément du code,
pas « finie » d'un bloc la veille de l'échéance. Le troisième, lui, est bien
un seul commit : retirer le fichier et empêcher son retour sont un même geste
de réparation.

### Le défi optionnel : ce que l'on coche

| Fichier | À cocher ? | Qui le fournit au correcteur |
|---|---|---|
| `notebook.ipynb` | Oui | Vous, c'est le rendu |
| `REFLEXION.md` | Oui | Vous, c'est le rendu |
| `donnees.py` | Inutile | La première cellule du notebook le retélécharge. Le commiter n'est pas une faute, mais il est redondant |
| `devoir1_online_retail.csv` | **Jamais** | Le notebook le télécharge. Et le `.gitignore` du dépôt devrait déjà l'avoir caché à GitHub Desktop : s'il apparaît dans `Changes`, c'est que le `.gitignore` a été modifié |

## Notebook 2, hack-time 2 : le tutoriel du devoir 1

Cet exercice n'a pas de corrigé au sens habituel : soit le commit est sur
GitHub, soit il n'y est pas. La table de dépannage du tutoriel couvre les
blocages habituels ; en voici les causes les plus fréquentes, vues de
l'enseignant.

| Symptôme | Cause | Solution |
|---|---|---|
| Pas d'invitation au dépôt du devoir 1 | Formulaire rempli après la création des dépôts, ou pseudo mal tapé | Vérifier son pseudo, corriger le formulaire si besoin, et lever la main : l'enseignant relance la création, qui ne touche pas aux dépôts existants |
| GitHub Desktop ne propose pas le dépôt au clonage | Invitation pas acceptée, ou liste pas rafraîchie | Accepter l'invitation sur github.com, puis dans GitHub Desktop fermer et rouvrir `Clone repository`. On peut aussi coller l'adresse du dépôt dans l'onglet `URL` |
| `Changes` reste vide après le téléchargement | Le fichier a été enregistré ailleurs que dans le dossier du dépôt, ou sous un autre nom (`notebook (1).ipynb`) | Déplacer le fichier dans le dossier, le renommer `notebook.ipynb` en remplaçant l'ancien |
| `Push origin` refusé : le dépôt distant a des commits plus récents | Un fichier a été modifié sur github.com, par exemple `REFLEXION.md` au crayon | `Fetch origin`, puis `Pull origin`, puis repousser |
| Un `.csv` apparaît dans `Changes` | Les données ont été téléchargées dans le dossier du dépôt | Ne pas le cocher, et supprimer le fichier : le notebook les retélécharge |

### Le défi optionnel : pourquoi les notebooks sont-ils pénibles à versionner ?

En ouvrant le diff de votre commit sur GitHub, vous avez vu que changer deux
mots produit un diff de centaines de lignes. La raison tient au format.

**Un `.ipynb` est un fichier JSON**, pas du texte. Il contient le code, mais
aussi les sorties, les images encodées en base64, les numéros d'exécution et des
métadonnées. Trois conséquences :

- **relancer un notebook sans rien changer produit un diff**, parce que les
  numéros d'exécution `[1]`, `[2]`, `[3]` ont bougé ;
- **une image générée par matplotlib pèse des milliers de lignes en base64**,
  illisibles et impossibles à fusionner ;
- **deux personnes qui modifient le même notebook créent presque toujours un
  conflit**, y compris si elles ont travaillé sur des cellules différentes.

C'est exactement l'argument développé en 2.3 : le notebook est un excellent
outil pour explorer et pour montrer, et un mauvais format pour versionner et
collaborer. D'où la règle : dès qu'un bout de code est stabilisé, il part dans un
fichier `.py`, où le diff redevient lisible.

Pour votre culture, deux outils traitent ce problème dans la vraie vie :
**`nbstripout`**, qui retire automatiquement les sorties avant chaque commit, et
**Jupytext**, qui maintient en parallèle une version `.py` du notebook. Aucun des
deux n'est nécessaire pour ce cours.